# TRIBE v2: simulate cortical activity from video, audio, or text

Colab workflow for Meta's TRIBE v2 model. In Colab, choose **Runtime → Change runtime type → A100 GPU** (40 GB or larger) and enable High RAM. This is a research model, not a clinical tool.

For video inference, place an `.mp4` in `/content/downloads/` (the notebook includes an uploader). Accept access to the gated `meta-llama/Llama-3.2-3B` model on Hugging Face and store a read token as the Colab secret `HF_TOKEN`.

## 1. Verify the GPU

TRIBE v2 needs an A100-class GPU: a T4 (16 GB) will run out of VRAM when its feature extractors load.

In [ ]:
import subprocess
import torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True, check=False)
print(result.stdout.strip() or result.stderr.strip())

assert torch.cuda.is_available(), (
    'No CUDA GPU detected. In Colab select Runtime → Change runtime type → A100 GPU.')
props = torch.cuda.get_device_properties(0)
MIN_VRAM_BYTES = 40_000_000_000  # 40 GB, not the guide's inconsistent 30 GB check
assert props.total_memory >= MIN_VRAM_BYTES, (
    f'Need ≥40 GB VRAM. Got {props.total_memory / 1e9:.1f} GB. Switch to an A100.')
print(f'GPU: {props.name} — {props.total_memory / 1e9:.1f} GB')

## 2. Pin NumPy, then restart Colab

`neuralset` currently requires NumPy < 2.1. Run the next cell once. If it installs NumPy, use **Runtime → Restart session**, then continue at the following cell. This restart is essential—Python cannot safely switch its loaded NumPy binary in place.

In [ ]:
import subprocess, sys
from importlib.metadata import version
from packaging.version import Version

installed = Version(version('numpy'))
if installed >= Version('2.1.0'):
    print(f'Pinning NumPy (currently {installed}) to >=1.26.4,<2.1.0 …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
                    'numpy>=1.26.4,<2.1.0'], check=True)
    raise RuntimeError('NumPy was changed. Restart the Colab session now, then run the next cell.')
print(f'NumPy {installed} is already compatible.')

In [ ]:
import numpy as np
from packaging.version import Version
assert Version(np.__version__) < Version('2.1.0'), (
    f'NumPy is {np.__version__}. Run the preceding cell and restart the runtime first.')
print(f'NumPy {np.__version__} checked')

!pip install -q 'tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git'
!pip install -q 'gradio>=4.19.0' 'nilearn>=0.10.3' 'plotly>=5.18.0' 'huggingface_hub>=0.25.0'

## 3. Hugging Face authentication and reliable model downloads

In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')

if token:
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)
    print('HF_TOKEN loaded.')
else:
    print('No Colab HF_TOKEN secret found; enter a Hugging Face read token below.')
    login()

os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '300'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '300'
os.environ['HF_HUB_HTTP_TIMEOUT'] = '300'

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

CACHE_DIR = Path('/content/tribe_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Pre-downloading Llama 3.2-3B (~6 GB; resumable) …')
snapshot_download(
    repo_id='meta-llama/Llama-3.2-3B',
    cache_dir=str(CACHE_DIR / 'llama'),
    ignore_patterns=['*.bin'],
)
print('Llama cache ready.')

## 4. Load TRIBE v2

In [ ]:
from tribev2 import TribeModel
import torch

print('Loading TRIBE v2 (first run downloads its checkpoint) …')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=str(CACHE_DIR), device='cuda')
used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded. VRAM after load: {used:.1f} / {total:.1f} GB')

## 5. Brain visualization helpers

In [ ]:
import numpy as np
from nilearn import datasets as nl_datasets
from nilearn.plotting import view_surf
from IPython.display import display, HTML

N_PER_HEMI = 10242
print('Fetching fsaverage5 mesh …')
fsavg = nl_datasets.fetch_surf_fsaverage(mesh='fsaverage5')
print('Mesh ready.')

def split_hemis(v):
    if len(v) == 2 * N_PER_HEMI:
        return v[:N_PER_HEMI], v[N_PER_HEMI:]
    return v[:len(v)//2], v[len(v)//2:]

def render_hemi(pred_vec, hemi='left', title=''):
    left, right = split_hemis(np.asarray(pred_vec))
    data = left if hemi == 'left' else right
    vmax = max(float(np.percentile(np.abs(data), 99)), 1e-6)
    return view_surf(fsavg[f'infl_{hemi}'], data, bg_map=fsavg[f'sulc_{hemi}'],
                     hemi=hemi, threshold='20%', cmap='hot', black_bg=True,
                     vmax=vmax, bg_on_data=True, colorbar=True, title=title)

def brain_html(pred_vec, title='', t=None):
    suffix = f' — t={t}s' if t is not None else ''
    left = render_hemi(pred_vec, 'left', f'{title} [Left]{suffix}')
    right = render_hemi(pred_vec, 'right', f'{title} [Right]{suffix}')
    return ('<div style="display:flex;gap:10px;background:#000;border-radius:10px">'
            f'<div style="flex:1">{left.get_iframe(width="100%", height="460px")}</div>'
            f'<div style="flex:1">{right.get_iframe(width="100%", height="460px")}</div></div>')

def show_brain(pred_vec, title='', t=None):
    display(HTML(brain_html(pred_vec, title, t)))

## 6. Text inference

In [ ]:
import tempfile

def text_to_preds(text):
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w', encoding='utf-8')
    try:
        tmp.write(text.strip()); tmp.flush(); os.fsync(tmp.fileno()); tmp.close()
        events = model.get_events_dataframe(text_path=tmp.name)
        preds, segments = model.predict(events=events)
        return np.asarray(preds), segments
    finally:
        if os.path.exists(tmp.name): os.unlink(tmp.name)

SAMPLE_TEXT = '''
The brain processes language through a distributed network in the left hemisphere.
Broca's area coordinates syntactic structure, while Wernicke's area handles semantics.
Together they form the language circuit activated when reading or hearing speech.
'''
preds, segments = text_to_preds(SAMPLE_TEXT)
print(f'Prediction shape: {preds.shape}')
t_show = min(5, len(preds) - 1)
show_brain(preds[t_show], title='Language stimulus', t=t_show)

## 7. Video inference from `downloads/`

Upload an MP4 if it is not already in `/content/downloads`; then select the video. A 15–30 second clip gives the model more useful context than a very short clip.

In [ ]:
from pathlib import Path
INPUT_DIR = Path('/content/downloads')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
videos = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() == '.mp4'])
if not videos:
    from google.colab import files
    print('No MP4 found. Choose a file to upload into /content/downloads/')
    for name, data in files.upload().items():
        (INPUT_DIR / name).write_bytes(data)
    videos = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() == '.mp4'])
assert videos, 'Add an .mp4 to /content/downloads/ and run this cell again.'
print('Available videos:')
for p in videos: print(' -', p.name)
VIDEO_PATH = videos[0]  # Change index/name here if you have multiple files.
print('Using:', VIDEO_PATH)

In [ ]:
video_events = model.get_events_dataframe(video_path=str(VIDEO_PATH))
video_preds, video_segments = model.predict(events=video_events)
video_preds = np.asarray(video_preds)
print(f'Video prediction shape: {video_preds.shape}')
t_show = min(5, len(video_preds) - 1)
show_brain(video_preds[t_show], title=VIDEO_PATH.name, t=t_show)

## 8. Language versus visual/spatial comparison

In [ ]:
TEXT_A = '''She spoke slowly and clearly, her voice filling the quiet room. Every sentence carried meaning, and each word was chosen with care. Language connects us, the professor said, bridging minds across time.'''
TEXT_B = '''The canyon walls rose steeply, layers of red and orange sandstone. A hawk circled overhead, its wings barely moving in the thermal current. Shadows shifted as the sun tracked its arc across the open desert sky.'''
preds_a, _ = text_to_preds(TEXT_A)
preds_b, _ = text_to_preds(TEXT_B)
shared = min(len(preds_a), len(preds_b))
t_show = min(5, shared - 1)
show_brain(preds_a[t_show], title='Condition A: Language', t=t_show)
show_brain(preds_b[t_show], title='Condition B: Visual', t=t_show)
show_brain(preds_a[t_show] - preds_b[t_show], title='Contrast A − B', t=t_show)

import matplotlib.pyplot as plt
diff_norms = [np.linalg.norm(preds_a[i] - preds_b[i]) for i in range(shared)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5))
ax1.plot(np.abs(preds_a).mean(axis=1)[:shared], label='A: Language')
ax1.plot(np.abs(preds_b).mean(axis=1)[:shared], label='B: Visual')
ax1.set(title='Mean cortical activation over time', xlabel='Time (s)'); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(diff_norms, color='#f39c12'); ax2.fill_between(range(shared), diff_norms, alpha=.2, color='#f39c12')
ax2.set(title='||A − B|| difference over time', xlabel='Time (s)'); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Gradio explorer

The guide’s final `demo.launch()` call lacked an interface. This cell supplies the single-input explorer it describes and caches predictions so moving the timestep does not rerun inference.

In [ ]:
import gradio as gr
_pred_cache = {}

def infer_for_ui(modality, video, audio, text):
    key = (modality, str(video or ''), str(audio or ''), text or '')
    if key not in _pred_cache:
        if modality == 'video':
            if not video: raise gr.Error('Choose a video file.')
            evts = model.get_events_dataframe(video_path=video)
        elif modality == 'audio':
            if not audio: raise gr.Error('Choose an audio file.')
            evts = model.get_events_dataframe(audio_path=audio)
        else:
            if not (text or '').strip(): raise gr.Error('Enter text.')
            p, _ = text_to_preds(text)
            _pred_cache[key] = p
            return p
        p, _ = model.predict(events=evts)
        _pred_cache[key] = np.asarray(p)
    return _pred_cache[key]

def run_ui(modality, video, audio, text):
    p = infer_for_ui(modality, video, audio, text)
    t = min(5, len(p)-1)
    return gr.update(maximum=max(0, len(p)-1), value=t), brain_html(p[t], modality.title(), t)

def move_ui(t, modality, video, audio, text):
    p = infer_for_ui(modality, video, audio, text)
    t = min(int(t), len(p)-1)
    return brain_html(p[t], modality.title(), t)

with gr.Blocks() as demo:
    gr.Markdown('# TRIBE v2 cortical activity explorer')
    modality = gr.Radio(['text', 'video', 'audio'], value='text', label='Input modality')
    with gr.Row():
        video = gr.Video(label='Video', type='filepath')
        audio = gr.Audio(label='Audio', type='filepath')
    text = gr.Textbox(label='Text', lines=5, value=SAMPLE_TEXT)
    run = gr.Button('Run prediction', variant='primary')
    timestep = gr.Slider(0, 0, value=0, step=1, label='Timestep (seconds)')
    output = gr.HTML()
    run.click(run_ui, [modality, video, audio, text], [timestep, output])
    timestep.change(move_ui, [timestep, modality, video, audio, text], output)

demo.launch(share=True, debug=False, server_name='0.0.0.0')